In [1]:
%%capture
!pip install koreanize-matplotlib
import koreanize_matplotlib

In [2]:
from google.colab import userdata
api_key = userdata.get('cos_api_key')

In [3]:
import pandas as pd
exchange = pd.read_csv('/content/exchange_2000_2024.csv')

In [4]:
# TIME 컬럼을 datetime으로 변환 (필요 시)
exchange["TIME"] = pd.to_datetime(exchange["TIME"])  # 문자열 or Period → datetime

In [5]:
# 외환보유액 변화량 계산
exchange['외환보유액_변화'] = exchange['한국 외환 보유액'].diff()

In [6]:
exchange.columns

Index(['TIME', '원/달러환율', '한국 장기 시장 금리', '미국 장기 시장 금리', '중국 장기 시장 금리',
       '한국 단기 시장 금리', '미국 단기 시장 금리', '중국 단기 시장 금리', '전체 수출 총액', '미국 수출 총액',
       '중국 수출 총액', '전체 수입 총액', '미국 수입 총액', '중국 수입 총액', '소득 교역 조건 지수',
       '해외 직접 투자 금액', '경제 심리 지수', '한국 기준 금리', '미국 기준 금리', '중국 기준 금리',
       '한국 소비자 물가지수', '미국 소비자 물가지수', '중국 소비자 물가지수', '한국 외환 보유액', '미국 외환 보유액',
       '중국 외환 보유액', '한국 산업 생산 지수', '미국 산업 생산 지수', '한국 실업률', '미국 실업률',
       '한국 주가지수', '미국 주가지수', '중국 주가지수', 'WTI 유가', '두바이 유가', 'Brent 유가',
       '천연가스 가격', '유연탄 가격', '철광석 가격', '구리 가격', '알루미늄 가격', '니켈 가격', '아연 가격',
       '금 가격', '대두 가격', '옥수수 가격', '소맥 가격', '원당 가격', '원면 가격', '외국인 투자 금액',
       '한국 경상수지', '미국 경상수지', '중국 경상수지', '한국 상품수지', '미국 상품수지', '중국 상품수지',
       '한국 경제성장률', '미국 경제성장률', '중국 경제성장률', '한국 GDP', '미국 GDP', '중국 GDP',
       '한국 GDP 디플레이터', '한국 중앙정부 부채 비율', '미국 중앙정부 부채 비율', '외환보유액_변화'],
      dtype='object')

- 인플레이션율 계산

In [16]:
exchange["인플레이션율"] = exchange["한국 소비자 물가지수"].pct_change(periods=12) * 100

- 실질 gdp, 실질 gdp 성장률

In [17]:
exchange["실질GDP"] = exchange["한국 GDP"] / exchange["한국 GDP 디플레이터"] * 100

In [18]:
exchange["실질GDP성장률"] = exchange["실질GDP"].pct_change(periods=12) * 100

<ipython-input-18-b07eb2aa3600>:1: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  exchange["실질GDP성장률"] = exchange["실질GDP"].pct_change(periods=12) * 100


- 외국

In [14]:
from sklearn.preprocessing import StandardScaler
from scipy.stats import pearsonr

# 사용할 변수
foreign_vars = ['한국 외환 보유액', '한국 상품수지', '한국 경상수지']

# 환율 + 외국시장 변수들에서 결측치 제거한 subset 생성
subset = exchange[foreign_vars + ['원/달러환율']].dropna()

# 표준화 후 외국시장 지표 생성
scaler = StandardScaler()
subset['외국시장_지표'] = scaler.fit_transform(subset[foreign_vars]).mean(axis=1)

# 상관관계 분석
corr, pval = pearsonr(subset['외국시장_지표'], subset['원/달러환율'])
print(f"상관계수: {corr:.3f}, p-value: {pval:.4f}")

상관계수: 0.321, p-value: 0.0000


- 개인

In [19]:
from sklearn.preprocessing import StandardScaler
from scipy.stats import pearsonr

# 사용할 개인시장 변수
personal_vars = ['한국 실업률', '인플레이션율', '경제 심리 지수']

# 환율 + 개인시장 변수들에서 결측치 제거
subset = exchange[personal_vars + ['원/달러환율']].dropna()

# 표준화 후 개인시장 지표 생성
scaler = StandardScaler()
subset['개인시장_지표'] = scaler.fit_transform(subset[personal_vars]).mean(axis=1)

# 상관관계 분석
corr, pval = pearsonr(subset['개인시장_지표'], subset['원/달러환율'])
print(f"상관계수: {corr:.3f}, p-value: {pval:.4f}")

상관계수: -0.413, p-value: 0.0000


- 기업

In [21]:
from sklearn.preprocessing import StandardScaler
from scipy.stats import pearsonr

# 사용할 기업시장 변수
corporate_vars = ['WTI 유가', '두바이 유가', 'Brent 유가',
       '천연가스 가격', '유연탄 가격', '철광석 가격', '구리 가격', '알루미늄 가격', '니켈 가격', '아연 가격',
       '금 가격', '대두 가격', '옥수수 가격', '소맥 가격', '원당 가격', '원면 가격', '미국 주가지수']

# 환율 + 기업시장 변수들에서 결측치 제거
subset = exchange[corporate_vars + ['원/달러환율']].dropna()

# 표준화 후 기업시장 지표 생성
scaler = StandardScaler()
subset['기업시장_지표'] = scaler.fit_transform(subset[corporate_vars]).mean(axis=1)

# 상관관계 분석
corr, pval = pearsonr(subset['기업시장_지표'], subset['원/달러환율'])
print(f"상관계수: {corr:.3f}, p-value: {pval:.4f}")

상관계수: -0.052, p-value: 0.3723


- 정부

In [26]:
exchangee = pd.read_csv('/content/exchange_add_debt.csv')

In [27]:
exchangee.columns

Index(['TIME', '원/달러환율', '한국 장기 시장 금리', '미국 장기 시장 금리', '중국 장기 시장 금리',
       '한국 단기 시장 금리', '미국 단기 시장 금리', '중국 단기 시장 금리', '전체 수출 총액', '미국 수출 총액',
       '중국 수출 총액', '전체 수입 총액', '미국 수입 총액', '중국 수입 총액', '소득 교역 조건 지수',
       '해외 직접 투자 금액', '경제 심리 지수', '한국 기준 금리', '미국 기준 금리', '중국 기준 금리',
       '한국 소비자 물가지수', '미국 소비자 물가지수', '중국 소비자 물가지수', '한국 외환 보유액', '미국 외환 보유액',
       '중국 외환 보유액', '한국 산업 생산 지수', '미국 산업 생산 지수', '한국 실업률', '미국 실업률',
       '한국 주가지수', '미국 주가지수', '중국 주가지수', 'WTI 유가', '두바이 유가', 'Brent 유가',
       '천연가스 가격', '유연탄 가격', '철광석 가격', '구리 가격', '알루미늄 가격', '니켈 가격', '아연 가격',
       '금 가격', '대두 가격', '옥수수 가격', '소맥 가격', '원당 가격', '원면 가격', '외국인 투자 금액',
       '한국 경상수지', '미국 경상수지', '중국 경상수지', '한국 상품수지', '미국 상품수지', '중국 상품수지',
       '한국 경제성장률', '미국 경제성장률', '중국 경제성장률', '한국 GDP', '미국 GDP', '중국 GDP',
       '한국 GDP 디플레이터', '한국 중앙정부 부채 비율', '미국 중앙정부 부채 비율', '한국 10년 채권수익률',
       '미국 10년 채권수익률'],
      dtype='object')

In [31]:
from sklearn.preprocessing import StandardScaler
from scipy.stats import pearsonr

# 사용할 정부시장 변수
gov_vars = ['한국 10년 채권수익률', '미국 기준 금리', '한국 GDP', '미국 GDP']

# 환율 + 정부시장 변수들에서 결측치 제거
subset = exchangee[gov_vars + ['원/달러환율']].dropna()

# 표준화 후 정부시장 지표 생성
scaler = StandardScaler()
subset['정부시장_지표'] = scaler.fit_transform(subset[gov_vars]).mean(axis=1)

# 상관관계 분석
corr, pval = pearsonr(subset['정부시장_지표'], subset['원/달러환율'])
print(f"상관계수: {corr:.3f}, p-value: {pval:.4f}")

상관계수: 0.085, p-value: 0.1590
